# LlamaMed-3.1-8B-Reasoner-Agent -- Colab Quickstart

Runs the CLI agent end-to-end on a free Colab GPU runtime: colored streaming output, command history, slash commands, tool calls (clinical calculators, PDF search, PubMed, web fallback), and persistent session/long-term memory.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is enough).

## 1. Get the code
Either clone your GitHub repo, or upload the project zip and unzip it.

In [ ]:
# Option A: clone from GitHub (edit the URL if your repo path differs)
!git clone https://github.com/sufirumii/LlamaMed-3.1-8B-Reasoner-Agent.git
%cd LlamaMed-3.1-8B-Reasoner-Agent

In [ ]:
# Option B: instead, if you uploaded a zip to this Colab session, use this cell
# instead of Option A above:
# from google.colab import files
# uploaded = files.upload()  # pick the .zip
# import zipfile, glob
# zip_name = glob.glob('*.zip')[0]
# with zipfile.ZipFile(zip_name) as z:
#     z.extractall('.')
# %cd LlamaMed-3.1-8B-Reasoner-Agent-main

## 2. Install dependencies
GGUF backend + llama.cpp CUDA wheel, so no manual CUDA toolkit setup is needed.

In [ ]:
!pip install -q -e .
!pip install -q -r requirements.txt
# Prebuilt CUDA wheel for llama-cpp-python so GPU offload works without compiling from source
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

## 3. Configure
`config.colab.yaml` is a profile layered on top of `config.yaml` -- smaller context window, GPU offload, shorter max tokens, and data directories under `/content` so they survive across cells in this session. Edit `gguf_hf_repo`/`gguf_hf_file` below if your quant lives somewhere else.

In [ ]:
!sed -n '1,40p' config.colab.yaml

## 4. Pre-download the model weights
Does this once so the first `chat`/`ask` call doesn't stall on a multi-GB download.

In [ ]:
!python scripts/download_model.py gguf \
  --repo Rumiii/LlamaMed-3.1-8B-Reasoner-GGUF \
  --file LlamaMed-3.1-8B-Reasoner.Q4_K_M.gguf \
  --out models/

## 5. One-shot test (`ask`)
Good for a quick smoke test before opening the interactive chat.

In [ ]:
!llamamed-agent --profile colab ask "What's the CKD-EPI eGFR for a 62-year-old woman with creatinine 1.4?"

## 6. Interactive chat
Runs the full CLI: colored streaming output, arrow-key command history, slash commands (`/help`, `/tools`, `/history`, `/memory <query>`, `/policy`, `/clear`, `/sessions`), guardrail checks on every turn, and a persistent session you can resume later with `--session <id>` (shown in the session banner when it starts).

Colab notebooks run cells non-interactively by default, so run this in a **terminal** instead for the full interactive experience: Tools -> Terminal (Colab Pro), or open this same repo in a Codespace/local terminal. If you're on the free tier without a terminal, use the `ask` one-shot form above in a loop instead -- the guardrails, memory, and tool-calling all work identically either way.

```bash
!llamamed-agent --profile colab chat
```

## 7. Ingest a PDF and search it
Upload a PDF to this session first (folder icon on the left, or `files.upload()`), then:

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # pick a PDF
# import glob
# pdf_path = glob.glob('*.pdf')[0]
# !llamamed-agent --profile colab ingest {pdf_path}
# !llamamed-agent --profile colab ask "Summarize the key findings in {pdf_path}"

## 8. Inspect memory and guardrails
Long-term memory (past Q/A pairs) and session transcripts persist under `/content/llamamed_data/` for the life of this Colab runtime (they're wiped when the runtime disconnects -- copy them to Drive if you want them to survive).

In [ ]:
!ls -la /content/llamamed_data/sessions/ 2>/dev/null || echo "(no sessions yet -- run a chat/ask turn first)"

In [ ]:
# Optional: copy memory/session data to Google Drive so it survives a runtime reset
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/llamamed_data /content/drive/MyDrive/llamamed_data_backup